# Demonstration 2: macromolecular crowding of a reversible bimolecular reaction

**What this validates:** volume exclusion and reaction acceptance working *together*.
the only demonstration that exercises both at once.

**The physics.** Consider $A + B \rightleftharpoons C$ with the product's volume chosen
to equal the sum of the reactants',

$$\sigma_C^3 = \sigma_A^3 + \sigma_B^3,$$

so the reaction leaves the packing fraction $\xi_3$ **unchanged**. It still changes the
other three weighted densities: two excluded-volume centres become one, so $\xi_0$
(number), $\xi_1$ and $\xi_2$ (surface) all drop. In a crowded medium that is
entropically favourable, so **crowding drives association**. This is the Minton
macromolecular-crowding effect, and choosing a volume-conserving product isolates it
from the trivial volume term.

An inert crowder species $X$, which never reacts and feels no field, sets the
crowding level, so it can be varied independently of the reactants.

## Two different claims, tested differently

**The exact one.** Summing detailed balance over states gives a flux balance that
involves no approximation:

$$k_F \,\langle n_A n_B \,\pi_F\rangle \;=\; k_R \,\langle n_C \,\pi_R\rangle$$

**The approximate one.** The textbook statement
$\Delta \ln K = -\Delta(\beta F_{\rm ex})$ evaluates the work at the *mean*
composition. That is a mean-field step: the acceptance is exponential in a work that
fluctuates, so by Jensen's inequality the mean-field prediction is not exact, and it
over-predicts, increasingly so with crowding. We check the sign and the trend, not
equality, and the gap is itself the lesson.

## Symbols, and where they live in the code

Every symbol used below, with the variable that carries it. Energies are in units of
$k_BT$ throughout, so $\beta = 1/k_BT$ never appears explicitly: what the code calls a
free energy is already $\beta F$.

### Lattice and transport

| symbol | code | meaning |
|---|---|---|
| $d$ | `lattice.dim` | dimension, 2 or 3 |
| $h$ | `voxel_nm` | voxel edge length, nm |
| $V$ | `lattice.voxel_volume_nm3` | voxel volume $h^3$, nm$^3$ (cubic even in 2D) |
| $\tau$ | `tau_s` | timestep, s |
| $D$ | `D_um2_s` | diffusion coefficient, $\mu$m$^2$/s |
| $q$ | `sim.hop.q` | baseline per-direction hop probability, $q = D\tau/h^2$ |
| $n_s(v)$ | `state.counts[s, v]` | integer count of species $s$ in voxel $v$ |
| $\rho$ | | number density, here the mean occupancy per voxel |

### Fields and coupling

| symbol | code | meaning |
|---|---|---|
| $\psi_k(v)$ | `psi[k]` | static basis field $k$, supplied by you and never modified |
| $\gamma_{s,k}$ | `Species.gamma` | coupling of species $s$ to field $k$ |
| $\phi_s(v)$ | | the potential species $s$ feels, $\phi_s(v) = \sum_k \gamma_{s,k}\,\psi_k(v)$ |

### Hard spheres

| symbol | code | meaning |
|---|---|---|
| $\sigma_s$ | `sigma_nm` | hard-sphere diameter of species $s$, nm |
| $d\xi^{(k)}_s$ | `exclusion.dxi[s, k]` | per-particle increment, $d\xi^{(k)}_s = \frac{\pi}{6}\sigma_s^k / V$ |
| $\xi_k(v)$ | `exclusion.xi(counts)[k]` | weighted density $k$, for $k = 0,1,2,3$ |
| $\eta$ | | packing fraction, the same thing as $\xi_3$ |
| $\beta F_{\rm ex}$ | `vex.bfex(xi, V)` | White-Bear/BMCSL excess free energy of a voxel |
| $\mu_{\rm ex}$ | `mu_ex_carnahan_starling` | excess chemical potential, $k_BT$ |

### The formulas the code implements

**Weighted densities.** Fundamental-measure theory reduces a hard-sphere mixture to
four scalar fields. On a lattice each is linear in the integer counts:

$$\xi_k(v) = \sum_s n_s(v)\, d\xi^{(k)}_s,
\qquad d\xi^{(k)}_s = \frac{\pi}{6}\frac{\sigma_s^{\,k}}{V},
\qquad k = 0,1,2,3.$$

$\xi_3$ is the packing fraction. $\xi_0$, $\xi_1$ and $\xi_2$ carry number, radius and
surface, which is why a reaction that merges two spheres into one of equal *volume*
still changes the free energy.

**White-Bear / BMCSL excess free energy**, per voxel, in $k_BT$:

$$\beta F_{\rm ex} = V\left[
  -\frac{6}{\pi}\,\xi_0 \ln(1-\xi_3)
  + \frac{18}{\pi}\,\frac{\xi_1 \xi_2}{1-\xi_3}
  + \frac{6}{\pi}\,\frac{\xi_2^3}{\xi_3^2}
    \left(\frac{\xi_3}{(1-\xi_3)^2} + \ln(1-\xi_3)\right)\right].$$

The $\ln(1-\xi_3)$ in the third term is what makes this BMCSL rather than the
Rosenfeld-1989 form. The factor $V$ converts a free-energy density into a per-voxel
energy.

**The hop.** A particle of species $s$ moves from voxel $v$ to a neighbour
$v' = v + e_\delta$ with probability

$$p_\delta(v) = q\; B\!\left(u_\delta(v)\right),
\qquad B(u) = \frac{u}{e^u - 1},$$

where $B$ is the Scharfetter-Gummel (Wang-Peskin-Elston) factor and $u$ is the total
work of the move in $k_BT$:

$$u_\delta(v) = \underbrace{\phi_s(v') - \phi_s(v)}_{\text{field}}
  + \underbrace{\Big[F(\xi_{v'} + d\xi_s) - F(\xi_{v'})\Big]}_{\text{insertion at } v'}
  - \underbrace{\Big[F(\xi_{v}) - F(\xi_{v} - d\xi_s)\Big]}_{\text{removal at } v}.$$

The $-d\xi_s$ in the last bracket is the **self-exclusion**: the hopping particle must
not feel its own volume at the voxel it is leaving. $B(0) = 1$ exactly, so with no work
the hop probability is just $q$.

**Why $\tau$ is small.** $B(u) \to |u|$ as $u \to -\infty$, so a strongly downhill move
can carry a probability far above $q$. The per-direction probabilities plus the stay
probability must sum to at most 1, which gives

$$2\,d\,q\,\mu_{\rm ex}(\eta_{\max}) \le 1$$

rather than the bare $q \le 1/(2d)$. At $\eta = 0.5$, $\mu_{\rm ex} \approx 17\,k_BT$,
so the admissible $\tau$ is an order of magnitude below the CFL bound.
`suggest_tau` computes it.

### Reactions

| symbol | code | meaning |
|---|---|---|
| $k_F$, $k_R$ | `k_forward`, `k_reverse` | forward and reverse rate constants |
| $\Delta\nu_s$ | `Reaction.dnu` | change in the count of species $s$ when the reaction fires once |
| $\Delta\Phi$ | | total work of the reaction, field plus exclusion, in $k_BT$ |
| $\pi_F$, $\pi_R$ | `ReactionSet.acceptance` | Fröhner-Noé acceptance factors |
| $Q$ | | reaction quotient, $\langle n_C\rangle / \langle n_A n_B\rangle$ |
| $K_{\rm eq}$ | | equilibrium constant, the value $Q$ takes at equilibrium |

**Propensity and acceptance.** A reaction of order 2 has per-voxel propensity
$k_F\, n_A(v)\, n_B(v)$, multiplied by an acceptance factor

$$\pi = \min\!\left(1,\; e^{-\Delta\Phi}\right),
\qquad \Delta\Phi = \underbrace{\sum_s \Delta\nu_s\, \phi_s(v)}_{\text{field}}
  + \underbrace{\Big[F(n + \Delta\nu) - F(n)\Big]}_{\text{exclusion}}.$$

**Reversibility.** The reverse work is evaluated from the **post-reaction** count
vector, not the pre-reaction one:

$$\Delta\Phi_F \text{ from } n, \qquad \Delta\Phi_R \text{ from } n + \Delta\nu
\quad\Longrightarrow\quad \Delta\Phi_R = -\Delta\Phi_F .$$

That makes $\pi_F/\pi_R = e^{-\Delta\Phi_F}$ identically, so detailed balance holds by
construction rather than approximately. Evaluating the reverse from $n$ instead gives
$F(n - \Delta\nu) - F(n)$, which is not the negative of the forward work, and detailed
balance then breaks silently.

**Note on the quotient.** The denominator is $\langle n_A n_B\rangle$, the mean of the
product, not $\langle n_A\rangle\langle n_B\rangle$. The reactants are correlated
through the conservation law, and only the mean of the product satisfies the
detailed-balance relation.

### Extra symbols for this demonstration

| symbol | code | meaning |
|---|---|---|
| $X$ | `Species("X", ..., inert=True)` | inert crowder: excludes volume, never reacts, feels no field |
| $\xi_3^X$ | `sim.crowder_packing_fraction()` | packing fraction contributed by the crowder alone |
| $\sigma_C$ | `SIGMA_C` | product diameter, chosen so $\sigma_C^3 = \sigma_A^3 + \sigma_B^3$ |

**The exact check** is a flux balance. Summing detailed balance over states introduces
no approximation:

$$k_F \,\big\langle n_A n_B \,\pi_F\big\rangle
  \;=\; k_R \,\big\langle n_C \,\pi_R\big\rangle .$$

**The physical claim** is the Minton crowding shift. Choosing a volume-conserving
product leaves $\xi_3$ unchanged by the reaction, so the shift comes from $\xi_0$,
$\xi_1$ and $\xi_2$ alone: two exclusion centres become one, which is entropically
favourable in a crowded medium.

**The mean-field prediction**, reported but *not* used as a passing criterion:

$$\Delta \ln K_{\rm eq} \;\approx\; -\,\Delta\!\left(\beta F_{\rm ex}\right)
  \Big|_{\text{mean composition}}.$$

It over-predicts, increasingly so with crowding, for two reasons the notebook returns
to at the end.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from vex_rddme import Simulation, Species
from vex_rddme.guards import suggest_tau
from vex_rddme import viz
from vex_rddme.observe import Series, QuotientAccumulator, report_comparison

In [ ]:
SIGMA_A = SIGMA_B = 6.0
SIGMA_C = (SIGMA_A ** 3 + SIGMA_B ** 3) ** (1 / 3)   # volume-conserving product
SIGMA_X = 7.0                                        # inert crowder
SHAPE, VOXEL_NM, CAP = (24, 24), 20.0, 24
K_F, K_R = 60.0, 300.0
N_PER_VOXEL = 3
CROWDER_COUNTS = (0, 2, 4, 6)
N_STEPS      = 40_000
BURN_IN      = N_STEPS // 3
SAMPLE_EVERY = 25

TAU_S = suggest_tau(D_um2_s=1.0, voxel_nm=VOXEL_NM, dim=2, eta_max=0.45)

print(f"sigma_A = sigma_B = {SIGMA_A}, sigma_C = {SIGMA_C:.3f}")
print(f"volume check: {SIGMA_A**3:.0f} + {SIGMA_B**3:.0f} = {SIGMA_C**3:.0f}  "
      f"-> the reaction leaves xi_3 unchanged")
print(f"ideal (uncrowded) K = k_F/k_R = {K_F/K_R}")
print(f"tau = {TAU_S:.3e} s")

## Sweep the crowder

Each run is an independent equilibration at one crowder density. For each we record the
reaction quotient, the flux-balance terms, and the mean composition.

In [ ]:
def run(n_x):
    species = [
        Species("A", SIGMA_A, np.zeros(1)),
        Species("B", SIGMA_B, np.zeros(1)),
        Species("C", SIGMA_C, np.zeros(1)),
        Species("X", SIGMA_X, np.zeros(1), inert=True),   # never reacts, never drifts
    ]
    sim = Simulation(
        shape=SHAPE, voxel_nm=VOXEL_NM, species=species, occupancy_cap=CAP,
        psi=np.zeros((1,) + SHAPE), D_um2_s=1.0, tau_s=TAU_S, seed=11,
        attach_log_handler=False,
    )
    sim.add_reaction("assoc", ["A", "B"], ["C"], K_F, K_R, typical_reactant_product=9.0)
    sim.set_counts("A", np.full(SHAPE, N_PER_VOXEL, dtype=np.int64))
    sim.set_counts("B", np.full(SHAPE, N_PER_VOXEL, dtype=np.int64))
    if n_x:
        sim.set_counts("X", np.full(SHAPE, n_x, dtype=np.int64))
    sim.record_initial()

    rs, rxn = sim.reactions, sim.reactions.reactions[0]
    acc = QuotientAccumulator(reactants=(0, 1), products=(2,))
    comp = Series("composition")
    fwd_flux = rev_flux = 0.0
    n = 0
    for i in range(N_STEPS):
        sim.step()
        if i >= BURN_IN and (i - BURN_IN) % SAMPLE_EVERY == 0:
            c = sim.state.counts
            acc.add(c)
            comp.add(c.mean(axis=1))
            pi_f = rs.acceptance(rs.total_work(c, rxn.dnu, rxn.didx_forward))
            post = c + rxn.dnu[:, None]
            pi_r = rs.acceptance(rs.total_work(post, -rxn.dnu, -rxn.didx_forward))
            fwd_flux += float((c[0] * c[1] * pi_f).mean())
            rev_flux += float((c[2] * pi_r).mean())
            n += 1
    sim.state.check_mass()
    return {
        "n_x": n_x,
        "eta_x": sim.crowder_packing_fraction(),
        "Q": float(acc.quotient[0]),
        "Q_sem": float(acc.sem[0]),
        "flux_f": K_F * fwd_flux / n,
        "flux_r": K_R * rev_flux / n,
        "work": sim.reactions.reaction_work_at(comp.mean),
    }

results = [run(n_x) for n_x in CROWDER_COUNTS]
print(f"{len(results)} runs complete")

## The exact check: flux balance

$k_F \langle n_A n_B \pi_F \rangle$ against $k_R \langle n_C \pi_R \rangle$. No
mean-field step, so this should agree to within sampling error at every crowder
density.

In [ ]:
print(f"{'n_X':>4} {'eta_X':>7} {'k_F<nAnB pi_F>':>15} {'k_R<nC pi_R>':>13} {'rel diff':>9}")
flux_residuals = []
for r in results:
    rel = abs(r["flux_f"] - r["flux_r"]) / max(r["flux_f"], r["flux_r"])
    flux_residuals.append(rel)
    print(f"{r['n_x']:>4} {r['eta_x']:>7.4f} {r['flux_f']:>15.3f} "
          f"{r['flux_r']:>13.3f} {rel*100:>8.2f}%")
print()
print(f"worst flux-balance residual: {max(flux_residuals)*100:.2f}%")

print()
print(report_comparison(
    "flux balance: k_R<nC pi_R> vs k_F<nAnB pi_F> (uncrowded)",
    results[0]["flux_r"], results[0]["flux_f"]))

## The physics: crowding drives association

In [ ]:
eta   = np.array([r["eta_x"] for r in results])
Q     = np.array([r["Q"] for r in results])
Q_sem = np.array([r["Q_sem"] for r in results])
work  = np.array([r["work"] for r in results])

dlnQ_measured  = np.log(Q) - np.log(Q[0])
dlnQ_meanfield = -(work - work[0])

print(f"{'eta_X':>7} {'Q':>8} {'+/-':>7} {'d lnQ meas':>11} {'mean-field':>11} {'ratio':>7}")
for i in range(len(results)):
    ratio = (dlnQ_measured[i] / dlnQ_meanfield[i]) if dlnQ_meanfield[i] else float("nan")
    print(f"{eta[i]:>7.4f} {Q[i]:>8.4f} {Q_sem[i]:>7.4f} {dlnQ_measured[i]:>11.4f} "
          f"{dlnQ_meanfield[i]:>11.4f} {ratio:>7.2f}")

monotonic = bool(np.all(np.diff(dlnQ_measured) > 0))
print()
print(f"K_eq rises monotonically with crowding: {monotonic}")
print(f"total shift over the sweep: d lnQ = {dlnQ_measured[-1]:+.4f} "
      f"({(Q[-1]/Q[0] - 1) * 100:+.1f}% in K)")

VERDICTS = {
    "flux balance (exact)": max(flux_residuals),
    "crowding raises K_eq": 0.0 if monotonic else 1.0,
}

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))

ax[0].errorbar(eta, Q, yerr=Q_sem, fmt="o-", ms=5, lw=1.4, capsize=3,
               color="#1f4e79", label="measured")
ax[0].axhline(K_F / K_R, ls="--", lw=1.2, color="#888",
              label=r"ideal $k_F/k_R$ (no crowding)")
ax[0].set_xlabel(r"crowder packing fraction $\xi_3^X$")
ax[0].set_ylabel(r"$Q = \langle n_C\rangle / \langle n_A n_B\rangle$")
ax[0].set_title("Crowding drives association (Minton)", fontsize=10)
ax[0].legend(frameon=False, fontsize=9)

ax[1].plot(eta, dlnQ_measured, "o-", ms=5, lw=1.4, color="#1f4e79", label="measured")
ax[1].plot(eta, dlnQ_meanfield, "s--", ms=4, lw=1.4, color="#c1440e",
           label=r"mean-field $-\Delta(\beta F_{\rm ex})$")
ax[1].set_xlabel(r"crowder packing fraction $\xi_3^X$")
ax[1].set_ylabel(r"$\Delta \ln K$")
ax[1].set_title("...and why mean-field over-predicts it", fontsize=10)
ax[1].legend(frameon=False, fontsize=9)

plt.tight_layout(); plt.show()

## What to take from this

**The exact relation holds.** The flux balance
$k_F\langle n_A n_B \pi_F\rangle = k_R\langle n_C \pi_R\rangle$ is satisfied to within
sampling error at every crowder density. That is the strongest statement available about
this machinery: it tests the exclusion work, the acceptance rule, and their coupling
simultaneously, with no approximation on the theory side.

**The physics is the expected one.** $K_{\rm eq}$ rises monotonically with crowder
packing fraction even though the reaction conserves volume. Two exclusion centres
becoming one frees up configurational entropy, and that shifts the equilibrium toward
the associated state.

**The textbook prediction over-predicts, and that is instructive.** The measured shift
is roughly half of $-\Delta(\beta F_{\rm ex})$ evaluated at the mean composition, and the
gap widens with crowding. Two reasons, both worth internalising:

1. **Jensen.** Acceptance is $e^{-\Delta\Phi}$, convex in a work that fluctuates from
   voxel to voxel. The mean of the exponential is not the exponential of the mean.
2. **The product crowds too.** $C$ is larger than either reactant, so as association
   proceeds the mixture's composition, and hence $\Delta(\beta F_{\rm ex})$ itself,
   changes along the way. Evaluating at a single mean composition cannot capture that.

A notebook that reported clean agreement here would be hiding one of these.

**Try changing:**

- `SIGMA_C = SIGMA_A`: a product *smaller* than the reactants' combined volume. Now
  $\xi_3$ drops as well, so the volume term adds to the number/surface term and the
  shift is larger.
- `SIGMA_C = (2 * SIGMA_A**3) ** (1/3) * 1.3`: a product bulkier than the reactants
  combined. The volume term now opposes association and can reverse the sign.
- `SIGMA_X = 3.0` at the same count: a much smaller crowder occupies far less volume,
  so the shift nearly vanishes. Crowding is about volume fraction, not particle number.